<div style="text-align: right;">
  <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/5/5e/HdM_Logo.svg/100px-HdM_Logo.svg.png"
       alt="HdM Logo"
       width="120">
</div>

# **Airbnb Europa: Eine Preisvorhersage durch lineare Regression**


Annette Kufner  
Modul "Data Analytics with Statistics"  
Januar 2026  
Prof. Dr. Jan Kirenz  
Hochschule der Medien Stuttgart  

### Zum Projekt

Motivation: 




Datenquelle: 
1. [*Insideairbnb*](https://insideairbnb.com "Inside Airbnb") ist ein unabhängiges, nicht-kommerzielles Daten- und Aktivismus-Projekt, welches untersucht wie die Plattform *Airbnb* in Wohngebieten eingesetzt wird. Es geht auf die Auswirkungen (u.a. von Kurzzeitvermietungen) auf Wohnraum, Gemeinschaften und Städte ein.
2. Die Daten sind jeweils als Momentaufnahme zu verstehen, da sie zu einem bestimmten Zeitpunkt durch Webscraping entstanden sind.
3. Die Daten (.csv-Dateien, Geodaten, ...) werden online zur Verfügung gestellt, um eine öffentliche Diskussion zu ermöglichen.
4. Unter [Data Assumptions](https://insideairbnb.com/data-assumptions/) wird auf die Datenqualität eingegangen und getroffene Maßnahmen für die gescrapten Daten festgehalten.





### 1.1.1 Datenbeschaffung und -aufbereitung

Auf der https://insideairbnb.com/get-the-data/ befinden sich Dateien für jede Stadt. Für die weitere Bearbeitung werden für Berlin diese Dateien gedownloadet.

1. `calendar.csv.gz`: detaillierte Kalenderdaten
2. `reviews.csv.gz`: detaillierte Prüfdaten
3. `reviews.csv`: Zusammenfassung der Überprüfungsdaten und der Listing-ID (zur Erleichterung zeitbasierter Analysen und Visualisierungen im Zusammenhang mit einem Listing).
5. `neighbourhoods.csv`: Nachbarschaftsliste für den Geofilter. Datenquelle: Stadt- oder Open-Source-GIS-Dateien.
6. `neighbourhoods.geojson`: GeoJSON-Datei der Stadtviertel.
3. `listings.csv`: Zusammenfassende Informationen und Kennzahlen für Immobilienangebote (gut geeignet für Visualisierungen).
7. `listings.csv.gz`: Detaillierte Angebotsdaten

Zudem besteht die Möglichkeit unter dem [Datenwörterbuch](https://docs.google.com/spreadsheets/d/1iWCNJcSutYqpULSQHlNyGInUvHg2BoUGoNRIGa6Szc4/edit?usp=sharing) alle Spalten teilweise mit und ohne Beschreibungen, für die jeweilige Datei (unterschiedliche Reiter in der Excel) einzusehen.
Die Dateien `neighbourhoods.csv`, `neighbourhoods.geojson`, `reviews.csv`, `reviews.csv.gz` und `calendar.csv.gz`, sind teilweise aufbereitete Dateien, welche korrespondierende Spalten in der `listings.csv.gz` bzw `listings.csv` aufweisen. Diese benötigen jedoch weitere Aufbereitung (Teil 2 und Teil 3) und werden im Folgenden nacheinander eingelesen und in ein *DataFrame* umgewandelt.

Datenbank: 
Von den über 50 Datensätzen zu europäischen Städten wurden 10 ausgesucht und so bereinigt, dass die lineare Regression durchgeführt werden kann. Zwei der jeweils sieben Daten pro Stadt wurden dafür verwendet 

Vorgehensweise: 



## Modell A: Verbindung Datenbank und Log-Transformation 'price'

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# --------------------------
# SCHRITT 1: Verbindung zur Datenbank herstellen
# --------------------------

# ERSETZEN SIE DIESE PLATZHALTER mit Ihren tatsächlichen Werten
DB_USER = 'Ihr_PostgreSQL_Benutzernname'
DB_PASSWORD = 'Ihr_Passwort'
DB_HOST = 'localhost'
DB_PORT = '5432'
DB_NAME = 'Ihr_Datenbankname'
TABELLE_NAME = 'listings_gesamt' # Angepasst an den Namen im .py-Skript

# Erstellen der Engine/Verbindungszeichenkette
engine = create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

# --------------------------
# SCHRITT 2: Daten mit SQL-Abfrage laden (INKL. ROHPREIS UND GEODATEN!)
# --------------------------

sql_query = f"""
SELECT 
    id, 
    price,                      -- NEU: Rohpreis wird geladen
    accommodates, 
    bedrooms, 
    beds,                       -- Hinzugefügt: Für konsistenten Prädiktor-Set
    room_type, 
    city, 
    latitude,                   -- NEU: Geodaten für Feature Engineering
    longitude,                  -- NEU: Geodaten für Feature Engineering
    review_scores_rating 
FROM 
    {TABELLE_NAME};             -- Die WHERE-Klausel entfällt, da das Cleaning-Skript bereits alles bereinigt hat
"""

# Führt die SQL-Abfrage aus und lädt das Ergebnis direkt in einen Pandas DataFrame
try:
    df_ml = pd.read_sql_query(sql_query, engine)
    print(f"Datenbankverbindung erfolgreich. {len(df_ml)} Einträge geladen.")

    # --------------------------
    # SCHRITT 3: Finale Vorbereitung im Notebook (Log-Transformation und Encoding)
    # --------------------------
    
    # 1. LOG-TRANSFORMATION HIER DURCHFÜHREN! (Zielvariable für Modell A)
    df_ml['ln_price'] = np.log(df_ml['price'])
    print("\n✅ Logarithmische Transformation ('ln_price') erstellt.")

    # 2. Imputation der Prädiktoren (z.B. fehlende Schlafzimmer-Angaben)
    # Diese logische Abfolge muss VOR der Regression erfolgen
    for col in ['bedrooms', 'beds', 'review_scores_rating']:
        if col in df_ml.columns and df_ml[col].isnull().any():
            median_val = df_ml[col].median()
            df_ml[col].fillna(median_val, inplace=True)
    print("✅ Imputation der numerischen Prädiktoren abgeschlossen.")

    # 3. Feature Engineering Geodaten (Platzhalter, muss noch implementiert werden)
    # df_ml['distance_to_center_km'] = berechne_distanz(df_ml, ...)
    
    # 4. One-Hot-Encoding starten (Prädiktoren für die Regression)
    df_ml = pd.get_dummies(df_ml, columns=['room_type', 'city'], drop_first=True)
    
    # Anzeigen der ersten Zeilen und der neuen Spalten
    print("\nDaten zur Modellierung bereit (Log-Modell-Pfad):")
    print(df_ml.head())
    
except Exception as e:
    print(f"FEHLER beim Laden der Daten aus PostgreSQL: {e}")
    print("Bitte prüfen Sie Ihre Verbindungsdaten (Benutzername/Passwort/DB-Name).")